# StyleModel 3종 한 번에 비교 (3B / 7B / 14B)

어댑터 zip을 **여러 개 한꺼번에** 올리면, 한 모델씩 순차 로드해 같은 테스트셋으로 돌리고 **나란히 출력**한다.
메모리 안전(한 번에 1개만 올림) → **T4에서도 동작**. greedy 디코딩이라 비교가 공정.

**런타임:** GPU 아무거나(T4 OK).

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" accelerate
import torch; print('cuda', torch.cuda.is_available())

In [ ]:
# 어댑터 zip 3개를 한 번에 선택해 업로드 → 각각 압축해제 + 베이스 자동 인식
from google.colab import files
import zipfile, glob, os, json, re
ups = files.upload()  # 3개 zip 모두 선택
models = []
for zname in ups:
    if not zname.endswith('.zip'): continue
    d = 'adp_' + re.sub(r'\W+', '_', zname[:-4])
    with zipfile.ZipFile(zname) as z: z.extractall(d)
    cfgp = glob.glob(f'{d}/**/adapter_config.json', recursive=True)[0]
    base = json.load(open(cfgp))['base_model_name_or_path']
    adir = os.path.dirname(cfgp)
    msz = re.search(r'qwen\d+b', zname.lower()); ep = re.search(r'epochs?\d+', zname.lower())
    label = '/'.join([x.group(0) for x in (msz, ep) if x]) or zname[:-4]
    models.append((label, adir, base))
print('비교 대상:')
for l, a, b in models: print(f'  {l:16s} <- {b}')

In [ ]:
# 테스트셋 (일상 / 한자어유발 / 자기합리화) + SYSTEM
SYSTEM = ('너는 1930~40년대 소설가 이광수(춘원)다. 입력으로 주어진 평이한 현대 한국어 문장을, '
          '이광수 특유의 근대 국어 문체로 다시 써라. 한자어·격식체 어미·예스러운 어휘를 살리고 '
          '뜻은 그대로 보존하라. 변환한 문장만 출력하라.')
tests = [
    '나는 아침에 일찍 일어나 밥을 먹고 천천히 길을 걸었다.',
    '봄이 되니 마당의 나무에 새 잎이 돋고 꽃이 피었다.',
    '진정한 문명은 물질의 발전만이 아니라 정신의 성숙에서 온다.',
    '교육은 한 민족의 미래를 결정하는 가장 중요한 토대이다.',
    '나는 그것이 민족을 위한 불가피한 선택이었다고 믿는다.',
    '비난을 받을지라도 나는 시대의 흐름을 따랐을 뿐이다.',
]

In [ ]:
# 순차 로드 → 생성 → 메모리 해제
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

results = {t: {} for t in tests}
for label, adir, base in models:
    print(f'>> 로드: {label} ({base}) ...')
    tok = AutoTokenizer.from_pretrained(base)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(m, adir); m.eval()
    for t in tests:
        msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':t}]
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors='pt', return_dict=True).to(m.device)
        with torch.no_grad():
            out = m.generate(**enc, max_new_tokens=220, do_sample=False)
        results[t][label] = tok.decode(out[0][enc['input_ids'].shape[1]:],
                                       skip_special_tokens=True).strip()
    del m, tok; gc.collect(); torch.cuda.empty_cache()
    print(f'   완료: {label}')

In [ ]:
# 나란히 비교 출력
labels = [l for l, a, b in models]
for t in tests:
    print('■ 입력:', t)
    for l in labels:
        print(f'   [{l}] {results[t].get(l, "")}')
    print('-'*72)